# Models — Season 2025 Inspection

This notebook builds and compares:
- Rolling Average Baseline
- Linear Regression model

Scope:
- Season 2025 only
- Players provided for inspection
- Train: GW 1–9
- Validation: GW 10–11


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

import matplotlib.pyplot as plt


In [5]:
print("Loading datasets...")

df = pd.read_csv("../output/training_data_v6_no0min_trial.csv", encoding="utf-8-sig")
players_df = pd.read_csv("../data/Players for inspection.csv", encoding="utf-8-sig")

print(f"Main dataset rows: {len(df):,}")
print(f"Players for inspection: {len(players_df):,}")


Loading datasets...
Main dataset rows: 25,735
Players for inspection: 12


In [6]:
print("Filtering season 2025 & inspection players...")

# Keep only season 2025
df = df[df["season"] == "2025-26"].copy()

# Keep only players provided
df = df[df["Player UUID"].isin(players_df["Player UUID"])].copy()

print(f"Rows after season + player filter: {len(df):,}")


Filtering season 2025 & inspection players...
Rows after season + player filter: 126


In [7]:
OUT_DATA_DIR = Path("../output/data/")
OUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

INSPECTION_DATA_PATH = OUT_DATA_DIR / "data_for_inspection_2025.csv"

df.to_csv(INSPECTION_DATA_PATH, index=False, encoding="utf-8-sig")

print("✔ Saved inspection dataset:")
print(f"   → {INSPECTION_DATA_PATH}")
print(f"   → Rows: {len(df)}")
print(f"   → Players: {df['Player UUID'].nunique()}")


✔ Saved inspection dataset:
   → ..\output\data\data_for_inspection_2025.csv
   → Rows: 126
   → Players: 12


In [8]:

# CELL 4 — Player eligibility filtering (season 2025-26)

import pandas as pd
from pathlib import Path

print("Loading inspection dataset...")

DATA_PATH = Path("../output/data/data_for_inspection_2025.csv")
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print(f"Rows loaded: {len(df)}")
print(f"Players loaded: {df['Player UUID'].nunique()}")


# Build per-player season stats (2025-26)

print("\nComputing per-player eligibility stats...")

player_stats = (
    df.groupby("Player UUID")
      .agg(
          Avg_Minutes_Played=("Minutes Played", "mean"),
          Total_Games_Played=("Gameweek", "count")
      )
      .reset_index()
)


# Apply eligibility rules
eligible_players = player_stats[
    (player_stats["Avg_Minutes_Played"] >= 60) &
    (player_stats["Total_Games_Played"] >= 5)
]["Player UUID"]

print(f"Eligible players after filters: {eligible_players.nunique()}")


# Filter main dataframe
df_filtered = df[df["Player UUID"].isin(eligible_players)].copy()

print("\nEligibility filtering complete:")
print(f"Rows after filtering: {len(df_filtered)}")
print(f"Players after filtering: {df_filtered['Player UUID'].nunique()}")


Loading inspection dataset...
Rows loaded: 126
Players loaded: 12

Computing per-player eligibility stats...
Eligible players after filters: 12

Eligibility filtering complete:
Rows after filtering: 126
Players after filtering: 12


In [9]:
df = df_filtered.copy()

df = df.sort_values(["Player UUID", "Gameweek"]).reset_index(drop=True)

df["Target_NextGW"] = (
    df.groupby("Player UUID")["Total Points"]
      .shift(-1)
)

before = len(df)
df = df.dropna(subset=["Target_NextGW"]).reset_index(drop=True)

print(f"Removed rows with no target: {before - len(df)}")
print("Rows after target creation:", len(df))


Removed rows with no target: 12
Rows after target creation: 114


In [10]:
df["Split"] = np.where(df["Gameweek"] <= 9, "Train", "Validation")

print(df["Split"].value_counts())
print("Players:", df["Player UUID"].nunique())


Split
Train         103
Validation     11
Name: count, dtype: int64
Players: 12
